# VGGHeads to Multi-PIE 68 Mapping with InfAnFace

This notebook has two stages:

1. Run VGGHeads on the InfAnFace dataset and learn a data-driven mapping from VGGHeads mesh vertices to Multi-PIE 68 landmarks.
2. Reuse the learned mapping to export only the 68 selected landmarks for new images.

Expected InfAnFace structure:

```text
infanface/
├── images/*.png
└── labels/*.txt
```

Each label file must contain:

```text
class_idx
x1 y1
...
x68 y68
```


In [3]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import numpy as np
from tqdm import tqdm

from head_detector import HeadDetector


ModuleNotFoundError: No module named 'smplx'

In [ ]:
@dataclass
class VGGHeadsSample:
    """Container for one image with InfAnFace GT and VGGHeads predictions."""

    stem: str
    image_path: Path
    label_path: Path
    class_idx: int
    image_rgb: np.ndarray
    gt_landmarks_68: np.ndarray
    vgg_vertices_2d: np.ndarray
    head_score: float


def list_image_paths(image_dir: str | Path, suffixes: Sequence[str] = (".png", ".jpg", ".jpeg")) -> List[Path]:
    """
    List image files in a directory using a deterministic order.

    Args:
        image_dir: Directory containing input images.
        suffixes: Accepted image file suffixes.

    Returns:
        Sorted list of image paths.
    """
    image_dir = Path(image_dir)
    return sorted(path for path in image_dir.iterdir() if path.suffix.lower() in suffixes)


def load_infanface_label(label_path: str | Path) -> Tuple[int, np.ndarray]:
    """
    Load an InfAnFace label file.

    The expected format is:
        class_idx
        x1 y1
        ...
        x68 y68

    Args:
        label_path: Path to the label file.

    Returns:
        A tuple containing the class index and a landmark array with shape (68, 2).

    Raises:
        FileNotFoundError: If the label file does not exist.
        ValueError: If the label format is invalid.
    """
    label_path = Path(label_path)
    if not label_path.exists():
        raise FileNotFoundError(f"Label file not found: {label_path}")

    with label_path.open("r", encoding="utf-8") as file:
        lines = [line.strip() for line in file.readlines() if line.strip()]

    if len(lines) < 69:
        raise ValueError(f"Expected at least 69 non-empty lines, got {len(lines)} in {label_path}")

    try:
        class_idx = int(float(lines[0]))
    except ValueError as error:
        raise ValueError(f"Invalid class_idx in {label_path}: {lines[0]}") from error

    coordinates = []
    for line_number, line in enumerate(lines[1:69], start=2):
        values = line.split()
        if len(values) != 2:
            raise ValueError(f"Invalid coordinate line {line_number} in {label_path}: {line}")
        coordinates.append([float(values[0]), float(values[1])])

    landmarks = np.asarray(coordinates, dtype=np.float32)
    if landmarks.shape != (68, 2):
        raise ValueError(f"Expected landmarks with shape (68, 2), got {landmarks.shape} in {label_path}")

    return class_idx, landmarks


def select_best_head(predictions) -> object:
    """
    Select the highest-confidence VGGHeads detection.

    Args:
        predictions: PredictionResult returned by HeadDetector.

    Returns:
        Highest-confidence detected head.

    Raises:
        ValueError: If no head is detected.
    """
    if not predictions.heads:
        raise ValueError("No heads were detected.")
    return max(predictions.heads, key=lambda head: float(head.score))


def run_vggheads_prediction_on_image(
    detector: HeadDetector,
    image_path: str | Path,
    confidence_threshold: float = 0.5,
) -> Tuple[object, object]:
    """
    Run VGGHeads inference on one image and return the raw prediction object.

    VGGHeads already maps projected vertices back to the original image coordinates,
    so downstream mapping code must use head.vertices_3d[:, :2] directly.

    Args:
        detector: Initialized HeadDetector instance.
        image_path: Path to the input image.
        confidence_threshold: Minimum detection confidence.

    Returns:
        A tuple containing the VGGHeads PredictionResult and the selected head.
    """
    predictions = detector(str(image_path), confidence_threshold=confidence_threshold)
    best_head = select_best_head(predictions)
    return predictions, best_head


def run_vggheads_on_image(
    detector: HeadDetector,
    image_path: str | Path,
    confidence_threshold: float = 0.5,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Run VGGHeads inference on one image.

    VGGHeads already maps the output vertices back to the original image coordinates.

    Args:
        detector: Initialized HeadDetector instance.
        image_path: Path to the input image.
        confidence_threshold: Minimum detection confidence.

    Returns:
        A tuple containing:
            - original RGB image
            - projected VGGHeads vertices with shape (N, 2)
            - selected head score
    """
    predictions, best_head = run_vggheads_prediction_on_image(
        detector=detector,
        image_path=image_path,
        confidence_threshold=confidence_threshold,
    )
    vertices_2d = best_head.vertices_3d[:, :2].astype(np.float32)
    return predictions.original_image, vertices_2d, float(best_head.score)


In [ ]:
def draw_points(
    image_rgb: np.ndarray,
    points: np.ndarray,
    color_rgb: Tuple[int, int, int],
    radius: int,
    thickness: int = -1,
) -> np.ndarray:
    """
    Draw points on an RGB image.

    Args:
        image_rgb: Input RGB image.
        points: Point array with shape (N, 2).
        color_rgb: Point color in RGB format.
        radius: Circle radius.
        thickness: Circle thickness. Use -1 for filled circles.

    Returns:
        RGB image with drawn points.
    """
    output_image = image_rgb.copy()
    height, width = output_image.shape[:2]

    for x_coord, y_coord in points:
        if not np.isfinite(x_coord) or not np.isfinite(y_coord):
            continue
        x_int = int(round(float(x_coord)))
        y_int = int(round(float(y_coord)))
        if 0 <= x_int < width and 0 <= y_int < height:
            cv2.circle(output_image, (x_int, y_int), radius, color_rgb, thickness)

    return output_image


def draw_landmark_indices(
    image_rgb: np.ndarray,
    landmarks: np.ndarray,
    color_rgb: Tuple[int, int, int],
    font_scale: float = 0.35,
    show_indices: bool = True,
) -> np.ndarray:
    """
    Draw landmark indices beside landmarks when requested.

    Args:
        image_rgb: Input RGB image.
        landmarks: Landmark array with shape (N, 2).
        color_rgb: Text color in RGB format.
        font_scale: OpenCV font scale.
        show_indices: Whether to draw landmark index numbers.

    Returns:
        RGB image with optional landmark indices.
    """
    output_image = image_rgb.copy()
    if not show_indices:
        return output_image

    height, width = output_image.shape[:2]
    for landmark_idx, (x_coord, y_coord) in enumerate(landmarks):
        if not np.isfinite(x_coord) or not np.isfinite(y_coord):
            continue
        x_int = int(round(float(x_coord)))
        y_int = int(round(float(y_coord)))
        if 0 <= x_int < width and 0 <= y_int < height:
            cv2.putText(
                output_image,
                str(landmark_idx),
                (x_int + 3, y_int - 3),
                cv2.FONT_HERSHEY_SIMPLEX,
                font_scale,
                color_rgb,
                1,
                cv2.LINE_AA,
            )
    return output_image


def draw_point_labels(
    image_rgb: np.ndarray,
    points: np.ndarray,
    labels: Sequence[str | int],
    color_rgb: Tuple[int, int, int],
    font_scale: float = 0.3,
    show_labels: bool = True,
) -> np.ndarray:
    """
    Draw arbitrary point labels beside points when requested.

    Args:
        image_rgb: Input RGB image.
        points: Point array with shape (N, 2).
        labels: Labels to draw, one per point.
        color_rgb: Text color in RGB format.
        font_scale: OpenCV font scale.
        show_labels: Whether to draw labels.

    Returns:
        RGB image with optional labels.
    """
    output_image = image_rgb.copy()
    if not show_labels:
        return output_image

    points = np.asarray(points, dtype=np.float32)
    if len(points) != len(labels):
        raise ValueError(f"Expected one label per point, got {len(labels)} labels for {len(points)} points")

    height, width = output_image.shape[:2]
    for label, (x_coord, y_coord) in zip(labels, points):
        if not np.isfinite(x_coord) or not np.isfinite(y_coord):
            continue
        x_int = int(round(float(x_coord)))
        y_int = int(round(float(y_coord)))
        if 0 <= x_int < width and 0 <= y_int < height:
            cv2.putText(
                output_image,
                str(label),
                (x_int + 3, y_int - 3),
                cv2.FONT_HERSHEY_SIMPLEX,
                font_scale,
                color_rgb,
                1,
                cv2.LINE_AA,
            )
    return output_image


def draw_lines_between_points(
    image_rgb: np.ndarray,
    source_points: np.ndarray,
    target_points: np.ndarray,
    color_rgb: Tuple[int, int, int],
    thickness: int = 1,
) -> np.ndarray:
    """
    Draw line segments between paired points.

    Args:
        image_rgb: Input RGB image.
        source_points: Source point array with shape (N, 2).
        target_points: Target point array with shape (N, 2).
        color_rgb: Line color in RGB format.
        thickness: Line thickness.

    Returns:
        RGB image with line segments.
    """
    output_image = image_rgb.copy()
    for source, target in zip(source_points, target_points):
        if not np.all(np.isfinite(source)) or not np.all(np.isfinite(target)):
            continue
        cv2.line(
            output_image,
            (int(round(float(source[0]))), int(round(float(source[1])))),
            (int(round(float(target[0]))), int(round(float(target[1])))),
            color_rgb,
            thickness,
            cv2.LINE_AA,
        )
    return output_image


def get_multipie_68_connections() -> list[list[int]]:
    """
    Return the standard MultiPIE/DLIB 68 landmark connection groups.

    Returns:
        A list of index groups defining the facial landmark topology.
    """
    return [
        list(range(0, 17)),
        list(range(17, 22)),
        list(range(22, 27)),
        list(range(27, 31)),
        list(range(31, 36)),
        [36, 37, 38, 39, 40, 41, 36],
        [42, 43, 44, 45, 46, 47, 42],
        [48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 48],
        [60, 61, 62, 63, 64, 65, 66, 67, 60],
    ]


def draw_multipie_68_landmarks(
    image_rgb: np.ndarray,
    landmarks_68: np.ndarray,
    point_color_rgb: tuple[int, int, int] = (0, 255, 0),
    line_color_rgb: tuple[int, int, int] | None = None,
    point_radius: int = 3,
    line_thickness: int = 2,
    show_indices: bool = False,
    index_color_rgb: tuple[int, int, int] = (255, 255, 255),
) -> np.ndarray:
    """
    Draw 68 MultiPIE/DLIB-style landmarks with optional index labels.

    Args:
        image_rgb: Input RGB image.
        landmarks_68: Landmark coordinates with shape (68, 2).
        point_color_rgb: RGB color used for landmark points.
        line_color_rgb: RGB color used for landmark connections. If None, point_color_rgb is used.
        point_radius: Radius of the landmark circles.
        line_thickness: Thickness of connection lines.
        show_indices: Whether to draw landmark indices.
        index_color_rgb: RGB color used for landmark index text.

    Returns:
        A copy of the image with landmarks drawn.
    """
    landmarks_68 = np.asarray(landmarks_68, dtype=np.float32)
    if landmarks_68.shape != (68, 2):
        raise ValueError(f"Expected landmarks_68 with shape (68, 2), got {landmarks_68.shape}")

    output_image = image_rgb.copy()
    line_color = point_color_rgb if line_color_rgb is None else line_color_rgb

    for connection_group in get_multipie_68_connections():
        for start_idx, end_idx in zip(connection_group[:-1], connection_group[1:]):
            start_point = landmarks_68[start_idx]
            end_point = landmarks_68[end_idx]
            if not np.all(np.isfinite(start_point)) or not np.all(np.isfinite(end_point)):
                continue
            cv2.line(
                output_image,
                (int(round(float(start_point[0]))), int(round(float(start_point[1])))),
                (int(round(float(end_point[0]))), int(round(float(end_point[1])))),
                line_color,
                line_thickness,
                cv2.LINE_AA,
            )

    output_image = draw_points(output_image, landmarks_68, color_rgb=point_color_rgb, radius=point_radius)
    output_image = draw_landmark_indices(
        output_image,
        landmarks_68,
        color_rgb=index_color_rgb,
        font_scale=0.35,
        show_indices=show_indices,
    )
    return output_image


def save_rgb_image(image_rgb: np.ndarray, output_path: str | Path) -> None:
    """
    Save an RGB image using OpenCV.

    Args:
        image_rgb: RGB image.
        output_path: Destination image path.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_path), cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))


def save_landmarks_txt(landmarks: np.ndarray, output_path: str | Path, include_class_idx: Optional[int] = None) -> None:
    """
    Save landmarks to a text file.

    Args:
        landmarks: Landmark array with shape (N, 2).
        output_path: Destination text path.
        include_class_idx: Optional class index to write as the first line.
    """
    landmarks = np.asarray(landmarks, dtype=np.float32)
    if landmarks.ndim != 2 or landmarks.shape[1] != 2:
        raise ValueError(f"Expected landmarks with shape (N, 2), got {landmarks.shape}")

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w", encoding="utf-8") as file:
        if include_class_idx is not None:
            file.write(f"{int(include_class_idx)}\n")
        for x_coord, y_coord in landmarks:
            file.write(f"{float(x_coord):.6f} {float(y_coord):.6f}\n")


def save_original_vggheads_overlay(
    predictions,
    output_path: Path,
    method: Optional[str] = None,
    convert_rgb_to_bgr: bool = False,
) -> None:
    """
    Save the native VGGHeads projected-head overlay before any 68-landmark mapping.

    The helper first tries the original repository behavior:
        result_image = predictions.draw()
        cv2.imwrite(str(output_path), result_image)

    Some VGGHeads checkpoints return only 68 projected vertices. In those cases,
    the native draw code can still request full-mesh indices such as 109 and fail.
    When that happens, this function saves a fallback overlay with every available
    projected vertex from head.vertices_3d[:, :2] so the mapping pipeline keeps
    running and the pre-mapping prediction can still be inspected.

    Args:
        predictions: PredictionResult returned by HeadDetector.
        output_path: Output image path.
        method: Optional drawing method accepted by predictions.draw. If None,
            predictions.draw() is called without arguments.
        convert_rgb_to_bgr: Whether to convert native draw output from RGB to BGR
            before saving. Keep False to match the original VGGHeads example.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        overlay_image = predictions.draw() if method is None else predictions.draw(method=method)
        if convert_rgb_to_bgr:
            overlay_image = cv2.cvtColor(overlay_image, cv2.COLOR_RGB2BGR)
        cv2.imwrite(str(output_path), overlay_image)
        return
    except Exception as error:
        print(
            f"[WARNING] Native VGGHeads draw failed for {output_path.name}; "
            f"saving projected vertices instead. Error: {error}"
        )

    fallback_rgb = predictions.original_image.copy()
    for head in predictions.heads:
        vertices_2d = head.vertices_3d[:, :2].astype(np.float32)
        fallback_rgb = draw_points(fallback_rgb, vertices_2d, color_rgb=(0, 255, 0), radius=1)

    cv2.imwrite(str(output_path), cv2.cvtColor(fallback_rgb, cv2.COLOR_RGB2BGR))


In [ ]:
def collect_vggheads_samples(
    dataset_root: str | Path,
    output_dir: str | Path,
    confidence_threshold: float = 0.5,
    model_name: str = "vgg_heads_l",
    save_overlays: bool = True,
    show_indices: bool = True,
    gt_color_rgb: Tuple[int, int, int] = (255, 0, 0),
    candidate_color_rgb: Tuple[int, int, int] = (0, 255, 255),
    save_original_overlays: bool = True,
    original_vggheads_method: Optional[str] = None,
) -> List[VGGHeadsSample]:
    """
    Run VGGHeads on all InfAnFace images and collect GT and predicted vertices.

    Args:
        dataset_root: Root directory containing images/ and labels/.
        output_dir: Directory where vertices and overlays will be saved.
        confidence_threshold: Minimum VGGHeads confidence threshold.
        model_name: VGGHeads model name accepted by HeadDetector.
        save_overlays: Whether to save GT and full-vertex debug overlays.
        show_indices: Whether debug overlays should draw landmark indices.
        gt_color_rgb: RGB color for ground-truth landmarks.
        candidate_color_rgb: RGB color for VGGHeads candidate vertices.
        save_original_overlays: Whether to save native VGGHeads overlays.
        original_vggheads_method: Optional drawing method accepted by predictions.draw. If None, use predictions.draw().

    Returns:
        List of collected samples.
    """
    dataset_root = Path(dataset_root)
    output_dir = Path(output_dir)
    image_dir = dataset_root / "images"
    label_dir = dataset_root / "labels"

    vertices_dir = output_dir / "vgg_vertices"
    overlay_dir = output_dir / "overlays_all_vertices"
    original_overlay_dir = output_dir / "original_vggheads_overlays"
    vertices_dir.mkdir(parents=True, exist_ok=True)
    if save_overlays:
        overlay_dir.mkdir(parents=True, exist_ok=True)
    if save_original_overlays:
        original_overlay_dir.mkdir(parents=True, exist_ok=True)

    detector = HeadDetector(model=model_name)
    image_paths = list_image_paths(image_dir)
    samples: List[VGGHeadsSample] = []
    failed_items: List[Tuple[str, str]] = []

    for image_path in tqdm(image_paths, desc="Running VGGHeads"):
        label_path = label_dir / f"{image_path.stem}.txt"
        if not label_path.exists():
            failed_items.append((image_path.name, "Missing label file"))
            continue

        try:
            class_idx, gt_landmarks = load_infanface_label(label_path)
            predictions, best_head = run_vggheads_prediction_on_image(
                detector=detector,
                image_path=image_path,
                confidence_threshold=confidence_threshold,
            )
            image_rgb = predictions.original_image
            vertices_2d = best_head.vertices_3d[:, :2].astype(np.float32)
            head_score = float(best_head.score)
        except Exception as error:
            failed_items.append((image_path.name, str(error)))
            continue

        np.savetxt(vertices_dir / f"{image_path.stem}.txt", vertices_2d, fmt="%.6f")

        if save_original_overlays:
            save_original_vggheads_overlay(
                predictions=predictions,
                output_path=original_overlay_dir / f"{image_path.stem}.png",
                method=original_vggheads_method,
            )

        if save_overlays:
            overlay = draw_points(image_rgb, vertices_2d, color_rgb=candidate_color_rgb, radius=1)
            overlay = draw_points(overlay, gt_landmarks, color_rgb=gt_color_rgb, radius=3)
            overlay = draw_landmark_indices(
                overlay,
                gt_landmarks,
                color_rgb=gt_color_rgb,
                font_scale=0.3,
                show_indices=show_indices,
            )
            save_rgb_image(overlay, overlay_dir / f"{image_path.stem}.png")

        samples.append(
            VGGHeadsSample(
                stem=image_path.stem,
                image_path=image_path,
                label_path=label_path,
                class_idx=class_idx,
                image_rgb=image_rgb,
                gt_landmarks_68=gt_landmarks,
                vgg_vertices_2d=vertices_2d,
                head_score=head_score,
            )
        )

    if failed_items:
        failure_path = output_dir / "failed_items.json"
        with failure_path.open("w", encoding="utf-8") as file:
            json.dump(failed_items, file, indent=2)
        print(f"[WARNING] Failed items saved to {failure_path}")

    print(f"Collected {len(samples)} valid samples out of {len(image_paths)} images.")
    return samples


In [ ]:
def compute_distance_tensor(samples: Sequence[VGGHeadsSample]) -> np.ndarray:
    """
    Compute distances from every GT landmark to every VGGHeads vertex for every image.

    Args:
        samples: Collected VGGHeads samples.

    Returns:
        Distance tensor with shape (num_images, 68, num_vertices).
    """
    if not samples:
        raise ValueError("At least one sample is required.")

    num_vertices = samples[0].vgg_vertices_2d.shape[0]
    for sample in samples:
        if sample.vgg_vertices_2d.shape[0] != num_vertices:
            raise ValueError("All samples must have the same number of VGGHeads vertices.")

    distances = []
    for sample in samples:
        gt_landmarks = sample.gt_landmarks_68.astype(np.float32)
        vertices = sample.vgg_vertices_2d.astype(np.float32)
        diff = gt_landmarks[:, None, :] - vertices[None, :, :]
        distances.append(np.linalg.norm(diff, axis=2))

    return np.stack(distances, axis=0)


def robust_distance_score(distance_tensor: np.ndarray, statistic: str = "median", trim_ratio: float = 0.1) -> np.ndarray:
    """
    Aggregate landmark-to-vertex distances across images.

    Args:
        distance_tensor: Distance tensor with shape (num_images, 68, num_vertices).
        statistic: Aggregation method. Options are "median", "mean", and "trimmed_mean".
        trim_ratio: Fraction removed from both tails when using trimmed mean.

    Returns:
        Score matrix with shape (68, num_vertices). Lower is better.
    """
    if statistic == "median":
        return np.median(distance_tensor, axis=0)

    if statistic == "mean":
        return np.mean(distance_tensor, axis=0)

    if statistic == "trimmed_mean":
        sorted_distances = np.sort(distance_tensor, axis=0)
        num_images = sorted_distances.shape[0]
        lower = int(num_images * trim_ratio)
        upper = num_images - lower
        if upper <= lower:
            raise ValueError("trim_ratio is too large for the number of images.")
        return np.mean(sorted_distances[lower:upper], axis=0)

    raise ValueError(f"Unsupported statistic: {statistic}")


def learn_index_mapping(
    samples: Sequence[VGGHeadsSample],
    statistic: str = "median",
    enforce_unique_vertices: bool = False,
    trim_ratio: float = 0.1,
) -> Dict[str, object]:
    """
    Learn a data-driven mapping from Multi-PIE 68 landmarks to VGGHeads vertex indices.

    Args:
        samples: Collected VGGHeads samples.
        statistic: Robust aggregation method over images.
        enforce_unique_vertices: Whether to prevent two landmarks from using the same vertex.
        trim_ratio: Trim ratio for trimmed mean.

    Returns:
        Mapping payload with selected vertex indices and quality statistics.
    """
    distance_tensor = compute_distance_tensor(samples)
    score_matrix = robust_distance_score(distance_tensor, statistic=statistic, trim_ratio=trim_ratio)

    if enforce_unique_vertices:
        from scipy.optimize import linear_sum_assignment

        landmark_indices, vertex_indices = linear_sum_assignment(score_matrix)
        ordered_indices = np.empty(68, dtype=np.int64)
        ordered_indices[landmark_indices] = vertex_indices
    else:
        ordered_indices = np.argmin(score_matrix, axis=1).astype(np.int64)

    selected_scores = score_matrix[np.arange(68), ordered_indices]
    per_image_errors = distance_tensor[:, np.arange(68), ordered_indices]

    payload: Dict[str, object] = {
        "mapping_type": "single_vertex_index",
        "statistic": statistic,
        "enforce_unique_vertices": bool(enforce_unique_vertices),
        "num_samples": len(samples),
        "vgg_indices_68": ordered_indices.astype(int).tolist(),
        "median_error_per_landmark": np.median(per_image_errors, axis=0).astype(float).tolist(),
        "mean_error_per_landmark": np.mean(per_image_errors, axis=0).astype(float).tolist(),
        "overall_median_error": float(np.median(per_image_errors)),
        "overall_mean_error": float(np.mean(per_image_errors)),
    }
    return payload


def learn_weighted_mapping(
    samples: Sequence[VGGHeadsSample],
    top_k: int = 5,
    statistic: str = "median",
    trim_ratio: float = 0.1,
) -> Dict[str, object]:
    """
    Learn a weighted local mapping from Multi-PIE landmarks to VGGHeads vertices.

    This is often more stable than forcing each landmark to use exactly one mesh vertex.

    Args:
        samples: Collected VGGHeads samples.
        top_k: Number of vertices used per landmark.
        statistic: Robust aggregation method over images.
        trim_ratio: Trim ratio for trimmed mean.

    Returns:
        Weighted mapping payload.
    """
    distance_tensor = compute_distance_tensor(samples)
    score_matrix = robust_distance_score(distance_tensor, statistic=statistic, trim_ratio=trim_ratio)

    mapping_items = []
    for landmark_idx in range(68):
        indices = np.argsort(score_matrix[landmark_idx])[:top_k]
        distances = score_matrix[landmark_idx, indices]
        weights = 1.0 / np.maximum(distances, 1e-6)
        weights = weights / weights.sum()
        mapping_items.append(
            {
                "landmark_index": landmark_idx,
                "indices": indices.astype(int).tolist(),
                "weights": weights.astype(float).tolist(),
                "scores": distances.astype(float).tolist(),
            }
        )

    return {
        "mapping_type": "weighted_vertices",
        "statistic": statistic,
        "top_k": int(top_k),
        "num_samples": len(samples),
        "items": mapping_items,
    }


def save_mapping(mapping_payload: Dict[str, object], output_path: str | Path) -> None:
    """
    Save a mapping payload to JSON.

    Args:
        mapping_payload: Mapping dictionary.
        output_path: Destination JSON path.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(mapping_payload, file, indent=2)


def load_mapping(mapping_path: str | Path) -> Dict[str, object]:
    """
    Load a mapping payload from JSON.

    Args:
        mapping_path: Mapping JSON path.

    Returns:
        Mapping dictionary.
    """
    with Path(mapping_path).open("r", encoding="utf-8") as file:
        return json.load(file)


In [ ]:
def get_top_k_candidates_for_sample(
    sample: VGGHeadsSample,
    top_k: int = 5,
) -> List[np.ndarray]:
    """
    Get the top-K nearest VGGHeads vertices for each GT landmark in one sample.

    Args:
        sample: Collected sample.
        top_k: Number of candidates per landmark.

    Returns:
        List with 68 arrays of vertex indices.
    """
    diff = sample.gt_landmarks_68[:, None, :] - sample.vgg_vertices_2d[None, :, :]
    distances = np.linalg.norm(diff, axis=2)
    return [np.argsort(distances[landmark_idx])[:top_k] for landmark_idx in range(68)]


def save_top_k_candidate_overlays(
    samples: Sequence[VGGHeadsSample],
    output_dir: str | Path,
    top_k: int = 5,
    show_indices: bool = True,
    gt_color_rgb: Tuple[int, int, int] = (255, 0, 0),
    candidate_color_rgb: Tuple[int, int, int] = (0, 255, 255),
) -> None:
    """
    Save overlays showing GT landmarks and their top-K nearest VGGHeads candidates.

    Args:
        samples: Collected samples.
        output_dir: Directory where overlays will be saved.
        top_k: Number of candidate vertices shown per GT landmark.
        show_indices: Whether debug overlays should draw landmark indices.
        gt_color_rgb: RGB color for ground-truth landmarks.
        candidate_color_rgb: RGB color for VGGHeads candidate vertices.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    for sample in tqdm(samples, desc="Saving top-K candidate overlays"):
        candidate_indices = get_top_k_candidates_for_sample(sample, top_k=top_k)
        candidate_points = np.concatenate([sample.vgg_vertices_2d[indices] for indices in candidate_indices], axis=0)
        candidate_labels = [int(index) for indices in candidate_indices for index in indices]

        overlay = draw_points(sample.image_rgb, candidate_points, color_rgb=candidate_color_rgb, radius=2)
        overlay = draw_point_labels(
            overlay,
            candidate_points,
            labels=candidate_labels,
            color_rgb=candidate_color_rgb,
            font_scale=0.22,
            show_labels=show_indices,
        )
        overlay = draw_points(overlay, sample.gt_landmarks_68, color_rgb=gt_color_rgb, radius=3)
        overlay = draw_landmark_indices(
            overlay,
            sample.gt_landmarks_68,
            color_rgb=gt_color_rgb,
            font_scale=0.3,
            show_indices=show_indices,
        )
        save_rgb_image(overlay, output_dir / f"{sample.stem}.png")


def apply_mapping_to_vertices(vertices_2d: np.ndarray, mapping_payload: Dict[str, object]) -> np.ndarray:
    """
    Apply a learned VGGHeads-to-Multi-PIE mapping to projected vertices.

    Args:
        vertices_2d: VGGHeads projected vertices with shape (N, 2).
        mapping_payload: Mapping dictionary loaded from JSON.

    Returns:
        Multi-PIE-style landmarks with shape (68, 2).
    """
    mapping_type = mapping_payload.get("mapping_type")

    if mapping_type == "single_vertex_index":
        indices = np.asarray(mapping_payload["vgg_indices_68"], dtype=np.int64)
        return vertices_2d[indices].astype(np.float32)

    if mapping_type == "weighted_vertices":
        landmarks = []
        for item in mapping_payload["items"]:
            indices = np.asarray(item["indices"], dtype=np.int64)
            weights = np.asarray(item["weights"], dtype=np.float32)
            point = np.sum(vertices_2d[indices] * weights[:, None], axis=0)
            landmarks.append(point)
        return np.asarray(landmarks, dtype=np.float32)

    raise ValueError(f"Unsupported mapping type: {mapping_type}")


def save_selected_mapping_overlays(
    samples: Sequence[VGGHeadsSample],
    mapping_payload: Dict[str, object],
    output_dir: str | Path,
    show_indices: bool = True,
    gt_color_rgb: Tuple[int, int, int] = (255, 0, 0),
    selected_color_rgb: Tuple[int, int, int] = (0, 255, 0),
    connector_color_rgb: Tuple[int, int, int] = (255, 255, 255),
) -> None:
    """
    Save overlays comparing GT landmarks against mapped VGGHeads 68 landmarks.

    Args:
        samples: Collected samples.
        mapping_payload: Learned mapping payload.
        output_dir: Directory where overlays will be saved.
        show_indices: Whether debug overlays should draw landmark indices.
        gt_color_rgb: RGB color for ground-truth landmarks.
        selected_color_rgb: RGB color for mapped selected landmarks.
        connector_color_rgb: RGB color for GT-to-selected connector lines.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    for sample in tqdm(samples, desc="Saving selected mapping overlays"):
        mapped_landmarks = apply_mapping_to_vertices(sample.vgg_vertices_2d, mapping_payload)
        overlay = draw_lines_between_points(
            sample.image_rgb,
            sample.gt_landmarks_68,
            mapped_landmarks,
            color_rgb=connector_color_rgb,
            thickness=1,
        )
        overlay = draw_points(overlay, sample.gt_landmarks_68, color_rgb=gt_color_rgb, radius=3)
        overlay = draw_points(overlay, mapped_landmarks, color_rgb=selected_color_rgb, radius=3)
        overlay = draw_landmark_indices(
            overlay,
            sample.gt_landmarks_68,
            color_rgb=gt_color_rgb,
            font_scale=0.3,
            show_indices=show_indices,
        )
        save_rgb_image(overlay, output_dir / f"{sample.stem}.png")


In [ ]:
def build_mapping_from_infanface(
    dataset_root: str | Path,
    output_dir: str | Path,
    confidence_threshold: float = 0.5,
    model_name: str = "vgg_heads_l",
    statistic: str = "median",
    top_k_candidates: int = 5,
    weighted_top_k: Optional[int] = None,
    enforce_unique_vertices: bool = False,
    show_indices: bool = True,
    gt_color_rgb: Tuple[int, int, int] = (255, 0, 0),
    candidate_color_rgb: Tuple[int, int, int] = (0, 255, 255),
    selected_color_rgb: Tuple[int, int, int] = (0, 255, 0),
    save_original_overlays: bool = True,
    original_vggheads_method: Optional[str] = None,
) -> Dict[str, object]:
    """
    End-to-end mapping builder for InfAnFace.

    Args:
        dataset_root: Root directory containing images/ and labels/.
        output_dir: Directory where all outputs will be saved.
        confidence_threshold: Minimum VGGHeads confidence threshold.
        model_name: VGGHeads model name.
        statistic: Aggregation method for distance scoring.
        top_k_candidates: Number of nearest candidates drawn in diagnostic overlays.
        weighted_top_k: If provided, learn a weighted mapping with this many vertices per landmark.
        enforce_unique_vertices: Whether single-index mapping must use unique vertices.
        show_indices: Whether mapping/debug overlays should draw landmark indices.
        gt_color_rgb: RGB color for ground-truth landmarks.
        candidate_color_rgb: RGB color for VGGHeads candidate vertices.
        selected_color_rgb: RGB color for mapped selected landmarks.
        save_original_overlays: Whether to save native VGGHeads overlays.
        original_vggheads_method: Optional drawing method accepted by predictions.draw. If None, use predictions.draw().

    Returns:
        Learned mapping payload.
    """
    output_dir = Path(output_dir)
    samples = collect_vggheads_samples(
        dataset_root=dataset_root,
        output_dir=output_dir,
        confidence_threshold=confidence_threshold,
        model_name=model_name,
        save_overlays=True,
        show_indices=show_indices,
        gt_color_rgb=gt_color_rgb,
        candidate_color_rgb=candidate_color_rgb,
        save_original_overlays=save_original_overlays,
        original_vggheads_method=original_vggheads_method,
    )

    save_top_k_candidate_overlays(
        samples=samples,
        output_dir=output_dir / f"overlays_top_{top_k_candidates}_candidates",
        top_k=top_k_candidates,
        show_indices=show_indices,
        gt_color_rgb=gt_color_rgb,
        candidate_color_rgb=candidate_color_rgb,
    )

    if weighted_top_k is None:
        mapping_payload = learn_index_mapping(
            samples=samples,
            statistic=statistic,
            enforce_unique_vertices=enforce_unique_vertices,
        )
        mapping_filename = "vggheads_to_multipie68_indices.json"
    else:
        mapping_payload = learn_weighted_mapping(
            samples=samples,
            top_k=weighted_top_k,
            statistic=statistic,
        )
        mapping_filename = f"vggheads_to_multipie68_weighted_k{weighted_top_k}.json"

    save_mapping(mapping_payload, output_dir / "mapping" / mapping_filename)

    save_selected_mapping_overlays(
        samples=samples,
        mapping_payload=mapping_payload,
        output_dir=output_dir / "overlays_selected_mapping",
        show_indices=show_indices,
        gt_color_rgb=gt_color_rgb,
        selected_color_rgb=selected_color_rgb,
    )

    print(json.dumps({key: mapping_payload[key] for key in mapping_payload if key not in {"items", "vgg_indices_68"}}, indent=2))
    return mapping_payload


In [ ]:
# Mapping/debug example: learn or validate the VGGHeads-to-MultiPIE mapping on InfAnFace.
# This mode keeps numbered overlays by default and saves the native VGGHeads overlays.
# Change these paths before running.

INFANFACE_ROOT = Path("/path/to/infanface")
OUTPUT_DIR = Path("/path/to/vggheads_infanface_mapping_outputs")

# Single-vertex mapping. This gives one VGGHeads mesh index per Multi-PIE landmark.
# mapping_payload = build_mapping_from_infanface(
#     dataset_root=INFANFACE_ROOT,
#     output_dir=OUTPUT_DIR,
#     confidence_threshold=0.5,
#     model_name="vgg_heads_l",
#     statistic="median",
#     top_k_candidates=5,
#     weighted_top_k=None,
#     enforce_unique_vertices=False,
#     show_indices=True,
#     gt_color_rgb=(255, 0, 0),
#     candidate_color_rgb=(0, 255, 255),
#     selected_color_rgb=(0, 255, 0),
#     save_original_overlays=True,
#     original_vggheads_method=None,
# )

# Weighted mapping. This is usually more robust because each landmark can be interpolated
# from a small local group of VGGHeads vertices.
# mapping_payload = build_mapping_from_infanface(
#     dataset_root=INFANFACE_ROOT,
#     output_dir=OUTPUT_DIR,
#     confidence_threshold=0.5,
#     model_name="vgg_heads_l",
#     statistic="median",
#     top_k_candidates=5,
#     weighted_top_k=5,
#     enforce_unique_vertices=False,
#     show_indices=True,
#     gt_color_rgb=(255, 0, 0),
#     candidate_color_rgb=(0, 255, 255),
#     selected_color_rgb=(0, 255, 0),
#     save_original_overlays=True,
#     original_vggheads_method=None,
# )


In [ ]:
def run_mapped_vggheads_single_image(
    image_path: str | Path,
    mapping_path: str | Path,
    output_image_path: str | Path,
    output_label_path: str | Path,
    confidence_threshold: float = 0.5,
    model_name: str = "vgg_heads_l",
    class_idx: Optional[int] = None,
    show_indices: bool = False,
    landmark_color_rgb: Tuple[int, int, int] = (0, 255, 0),
    line_color_rgb: Tuple[int, int, int] = (0, 255, 0),
    point_radius: int = 3,
    line_thickness: int = 2,
    save_original_overlay: bool = True,
    original_vggheads_overlay_path: str | Path | None = None,
    original_vggheads_method: Optional[str] = None,
) -> np.ndarray:
    """
    Run VGGHeads on one image and export only mapped Multi-PIE 68 landmarks.

    Args:
        image_path: Input image path.
        mapping_path: Path to a learned mapping JSON file.
        output_image_path: Path where the connected 68-landmark overlay will be saved.
        output_label_path: Path where the 68 landmarks will be saved.
        confidence_threshold: Minimum VGGHeads confidence threshold.
        model_name: VGGHeads model name.
        class_idx: Optional class index to write as first line in the label file.
        show_indices: Whether to draw landmark indices on the final inference overlay.
        landmark_color_rgb: RGB color for mapped landmark points.
        line_color_rgb: RGB color for mapped landmark connections.
        point_radius: Radius of landmark circles.
        line_thickness: Thickness of landmark connection lines.
        save_original_overlay: Whether to save the native VGGHeads overlay.
        original_vggheads_overlay_path: Optional path for the native VGGHeads overlay.
        original_vggheads_method: Optional drawing method accepted by predictions.draw. If None, use predictions.draw().

    Returns:
        Mapped landmarks with shape (68, 2).
    """
    detector = HeadDetector(model=model_name)
    mapping_payload = load_mapping(mapping_path)
    predictions, best_head = run_vggheads_prediction_on_image(
        detector=detector,
        image_path=image_path,
        confidence_threshold=confidence_threshold,
    )
    image_rgb = predictions.original_image
    vertices_2d = best_head.vertices_3d[:, :2].astype(np.float32)

    if save_original_overlay:
        if original_vggheads_overlay_path is None:
            output_image_path = Path(output_image_path)
            output_root = output_image_path.parent.parent if output_image_path.parent.name == "images" else output_image_path.parent
            original_vggheads_overlay_path = output_root / "original_vggheads_overlays" / output_image_path.name
        save_original_vggheads_overlay(
            predictions=predictions,
            output_path=Path(original_vggheads_overlay_path),
            method=original_vggheads_method,
        )

    landmarks_68 = apply_mapping_to_vertices(vertices_2d, mapping_payload)
    overlay = draw_multipie_68_landmarks(
        image_rgb=image_rgb,
        landmarks_68=landmarks_68,
        point_color_rgb=landmark_color_rgb,
        line_color_rgb=line_color_rgb,
        point_radius=point_radius,
        line_thickness=line_thickness,
        show_indices=show_indices,
    )

    save_rgb_image(overlay, output_image_path)
    save_landmarks_txt(landmarks_68, output_label_path, include_class_idx=class_idx)

    return landmarks_68


def run_mapped_vggheads_directory(
    image_dir: str | Path,
    mapping_path: str | Path,
    output_dir: str | Path,
    confidence_threshold: float = 0.5,
    model_name: str = "vgg_heads_l",
    class_idx: Optional[int] = None,
    show_indices: bool = False,
    landmark_color_rgb: Tuple[int, int, int] = (0, 255, 0),
    line_color_rgb: Tuple[int, int, int] = (0, 255, 0),
    point_radius: int = 3,
    line_thickness: int = 2,
    save_original_overlays: bool = True,
    original_vggheads_method: Optional[str] = None,
) -> None:
    """
    Run mapped VGGHeads inference on a directory of images.

    Args:
        image_dir: Directory containing input images.
        mapping_path: Path to learned mapping JSON file.
        output_dir: Output directory.
        confidence_threshold: Minimum VGGHeads confidence threshold.
        model_name: VGGHeads model name.
        class_idx: Optional class index to write as the first line in every label file.
        show_indices: Whether to draw landmark indices on final inference overlays.
        landmark_color_rgb: RGB color for mapped landmark points.
        line_color_rgb: RGB color for mapped landmark connections.
        point_radius: Radius of landmark circles.
        line_thickness: Thickness of landmark connection lines.
        save_original_overlays: Whether to save native VGGHeads overlays.
        original_vggheads_method: Optional drawing method accepted by predictions.draw. If None, use predictions.draw().
    """
    image_dir = Path(image_dir)
    output_dir = Path(output_dir)
    image_output_dir = output_dir / "images"
    label_output_dir = output_dir / "labels"
    original_overlay_dir = output_dir / "original_vggheads_overlays"
    image_output_dir.mkdir(parents=True, exist_ok=True)
    label_output_dir.mkdir(parents=True, exist_ok=True)
    if save_original_overlays:
        original_overlay_dir.mkdir(parents=True, exist_ok=True)

    detector = HeadDetector(model=model_name)
    mapping_payload = load_mapping(mapping_path)
    failed_items: List[Tuple[str, str]] = []

    for image_path in tqdm(list_image_paths(image_dir), desc="Running mapped VGGHeads"):
        try:
            predictions, best_head = run_vggheads_prediction_on_image(
                detector=detector,
                image_path=image_path,
                confidence_threshold=confidence_threshold,
            )
            image_rgb = predictions.original_image
            vertices_2d = best_head.vertices_3d[:, :2].astype(np.float32)
            landmarks_68 = apply_mapping_to_vertices(vertices_2d, mapping_payload)

            if save_original_overlays:
                save_original_vggheads_overlay(
                    predictions=predictions,
                    output_path=original_overlay_dir / f"{image_path.stem}.png",
                    method=original_vggheads_method,
                )

            overlay = draw_multipie_68_landmarks(
                image_rgb=image_rgb,
                landmarks_68=landmarks_68,
                point_color_rgb=landmark_color_rgb,
                line_color_rgb=line_color_rgb,
                point_radius=point_radius,
                line_thickness=line_thickness,
                show_indices=show_indices,
            )

            save_rgb_image(overlay, image_output_dir / f"{image_path.stem}.png")
            save_landmarks_txt(landmarks_68, label_output_dir / f"{image_path.stem}.txt", include_class_idx=class_idx)
        except Exception as error:
            failed_items.append((image_path.name, str(error)))

    if failed_items:
        with (output_dir / "failed_items.json").open("w", encoding="utf-8") as file:
            json.dump(failed_items, file, indent=2)
        print(f"[WARNING] Failed items: {len(failed_items)}")


In [ ]:
# Final inference example: apply a learned mapping to another image folder.
# This mode saves connected 68-landmark overlays without landmark numbers by default,
# exports 68-row txt labels, and also stores native VGGHeads overlays.
# Change these paths before running.

# run_mapped_vggheads_directory(
#     image_dir="/path/to/new/images",
#     mapping_path="/path/to/vggheads_to_multipie68_weighted_k5.json",
#     output_dir="/path/to/mapped_vggheads_outputs",
#     confidence_threshold=0.5,
#     model_name="vgg_heads_l",
#     class_idx=None,
#     show_indices=False,
#     landmark_color_rgb=(0, 255, 0),
#     line_color_rgb=(0, 255, 0),
#     save_original_overlays=True,
#     original_vggheads_method=None,
# )

# Single-image version:
# landmarks_68 = run_mapped_vggheads_single_image(
#     image_path="/path/to/new/image.png",
#     mapping_path="/path/to/vggheads_to_multipie68_weighted_k5.json",
#     output_image_path="/path/to/mapped_vggheads_outputs/images/image.png",
#     output_label_path="/path/to/mapped_vggheads_outputs/labels/image.txt",
#     confidence_threshold=0.5,
#     model_name="vgg_heads_l",
#     class_idx=None,
#     show_indices=False,
#     landmark_color_rgb=(0, 255, 0),
#     line_color_rgb=(0, 255, 0),
#     save_original_overlay=True,
# )


In [ ]:
def run_native_vggheads_repository_export_directory(
    image_dir: str | Path,
    output_dir: str | Path,
    confidence_threshold: float = 0.5,
    model_name: str = "vgg_heads_l",
    image_suffixes: Sequence[str] = (".png", ".jpg", ".jpeg"),
    save_meshes: bool = True,
    save_aligned_heads: bool = True,
) -> List[Dict[str, object]]:
    """
    Run the native VGGHeads repository export workflow on every image in a directory.

    This function intentionally does not use any learned VGGHeads-to-MultiPIE mapping.
    It mirrors the original repository workflow for each image:
        predictions = detector(image_path)
        result_image = predictions.draw()
        cv2.imwrite(..., result_image)
        predictions.save_meshes(...)
        aligned_heads = predictions.get_aligned_heads()

    Args:
        image_dir: Directory containing input images.
        output_dir: Root directory where native VGGHeads outputs will be saved.
        confidence_threshold: Minimum VGGHeads confidence threshold.
        model_name: VGGHeads model name accepted by HeadDetector.
        image_suffixes: Accepted image file suffixes.
        save_meshes: Whether to save native VGGHeads head mesh files.
        save_aligned_heads: Whether to save native aligned head crops.

    Returns:
        A list of per-image summary dictionaries.
    """
    image_dir = Path(image_dir)
    output_dir = Path(output_dir)
    overlay_dir = output_dir / "native_vggheads_overlays"
    mesh_root = output_dir / "native_vggheads_head_meshes"
    aligned_root = output_dir / "native_vggheads_aligned_heads"

    overlay_dir.mkdir(parents=True, exist_ok=True)
    if save_meshes:
        mesh_root.mkdir(parents=True, exist_ok=True)
    if save_aligned_heads:
        aligned_root.mkdir(parents=True, exist_ok=True)

    detector = HeadDetector(model=model_name)
    image_paths = list_image_paths(image_dir, suffixes=image_suffixes)
    summaries: List[Dict[str, object]] = []
    failed_items: List[Tuple[str, str]] = []

    for image_path in tqdm(image_paths, desc="Running native VGGHeads export"):
        try:
            predictions = detector(str(image_path), confidence_threshold=confidence_threshold)

            result_image = predictions.draw()
            result_path = overlay_dir / f"{image_path.stem}.png"
            cv2.imwrite(str(result_path), result_image)

            mesh_dir = None
            if save_meshes:
                mesh_dir = mesh_root / image_path.stem
                mesh_dir.mkdir(parents=True, exist_ok=True)
                predictions.save_meshes(str(mesh_dir))

            aligned_dir = None
            aligned_count = 0
            if save_aligned_heads:
                aligned_dir = aligned_root / image_path.stem
                aligned_dir.mkdir(parents=True, exist_ok=True)
                aligned_heads = predictions.get_aligned_heads()
                aligned_count = len(aligned_heads)
                for head_idx, aligned_head in enumerate(aligned_heads):
                    cv2.imwrite(str(aligned_dir / f"aligned_head_{head_idx}.png"), aligned_head)

            summary = {
                "image_path": str(image_path),
                "num_heads": len(predictions.heads),
                "result_image_path": str(result_path),
                "mesh_dir": str(mesh_dir) if mesh_dir is not None else None,
                "aligned_heads_dir": str(aligned_dir) if aligned_dir is not None else None,
                "num_aligned_heads": aligned_count,
                "status": "ok",
            }
            summaries.append(summary)

            print(f"Detected {len(predictions.heads)} heads in {image_path.name}.")
            print(f"Result image saved as '{result_path}'")
            if mesh_dir is not None:
                print(f"Head meshes saved in '{mesh_dir}' folder")
            if aligned_dir is not None:
                print(f"Aligned head crops saved in '{aligned_dir}' folder")
        except Exception as error:
            failed_items.append((image_path.name, str(error)))
            summaries.append(
                {
                    "image_path": str(image_path),
                    "num_heads": 0,
                    "result_image_path": None,
                    "mesh_dir": None,
                    "aligned_heads_dir": None,
                    "num_aligned_heads": 0,
                    "status": "failed",
                    "error": str(error),
                }
            )

    summary_path = output_dir / "native_vggheads_export_summary.json"
    with summary_path.open("w", encoding="utf-8") as file:
        json.dump(summaries, file, indent=2)

    if failed_items:
        failed_path = output_dir / "native_vggheads_failed_items.json"
        with failed_path.open("w", encoding="utf-8") as file:
            json.dump(failed_items, file, indent=2)
        print(f"[WARNING] Failed native VGGHeads exports saved to {failed_path}")

    print(f"Processed {len(image_paths)} images.")
    print(f"Native VGGHeads export summary saved to {summary_path}")
    return summaries


# Native VGGHeads repository export example.
# This is independent from the 68-landmark mapping code above.
# It saves predictions.draw(), native meshes, and aligned head crops for a full image directory.
# Change these paths before running.

# native_export_summaries = run_native_vggheads_repository_export_directory(
#     image_dir="/path/to/images",
#     output_dir="/path/to/native_vggheads_outputs",
#     confidence_threshold=0.5,
#     model_name="vgg_heads_l",
#     save_meshes=True,
#     save_aligned_heads=True,
# )
